In [1]:
import os
import glob
import numpy as np
import pandas as pd
import librosa
import noisereduce as nr
import warnings

warnings.filterwarnings('ignore')

# --- CONFIGURATION ---
DATASET_PATH = r"../data/raw/Audio/Baby Cry Dataset/" 
OUTPUT_PATH = r"../data/processed/"
OUTPUT_FILE = "Audio_CatBoost_Features.csv"

LABEL_MAP = {
    'hungry': 0,
    'belly pain': 1, 'cold_hot': 1, 'discomfort': 1,
    'tired': 2, 'burping': 2, 'lonely': 2, 'scared': 2
}

def extract_features_advanced(y, sr):
    features = []
    
    # 1. MFCC (Base)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    features.append(np.mean(mfcc, axis=1))
    features.append(np.std(mfcc, axis=1))
    
    # 2. Delta MFCC (Velocity - Rate of Change) <--- NEW SUPER FEATURE
    # This captures "How fast is the cry changing?"
    mfcc_delta = librosa.feature.delta(mfcc)
    features.append(np.mean(mfcc_delta, axis=1))
    features.append(np.std(mfcc_delta, axis=1))
    
    # 3. Delta-Delta MFCC (Acceleration)
    mfcc_delta2 = librosa.feature.delta(mfcc, order=2)
    features.append(np.mean(mfcc_delta2, axis=1))
    features.append(np.std(mfcc_delta2, axis=1))

    # 4. Mel Spectrogram
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64) # Reduced to 64 for stability
    features.append(np.mean(mel, axis=1))
    features.append(np.std(mel, axis=1))
    
    # 5. Chroma & Contrast
    stft = np.abs(librosa.stft(y))
    chroma = librosa.feature.chroma_stft(S=stft, sr=sr)
    contrast = librosa.feature.spectral_contrast(S=stft, sr=sr)
    
    features.append(np.mean(chroma, axis=1))
    features.append(np.mean(contrast, axis=1))
    
    return np.concatenate(features)

# --- MAIN LOOP ---
data_list = []

print("🚀 Starting Advanced Feature Extraction...")

for folder_name, label in LABEL_MAP.items():
    folder_path = os.path.join(DATASET_PATH, folder_name)
    files = glob.glob(os.path.join(folder_path, "*.wav"))
    
    print(f"   Processing {folder_name}...")
    
    for file in files:
        try:
            # Load 5s
            y, sr = librosa.load(file, duration=5.0, sr=22050)
            
            # Noise Reduction (Light)
            try:
                y = nr.reduce_noise(y=y, sr=sr, prop_decrease=0.6)
            except: pass

            # Extract
            feat = extract_features_advanced(y, sr)
            
            # Append Label to end
            row = np.append(feat, label)
            data_list.append(row)
            
            # --- AUGMENTATION (Balance Classes) ---
            # Smart Augmentation for Normal (2) and Hunger (0)
            if label in [0, 2]:
                # Noise
                noise_factor = 0.005
                y_noise = y + noise_factor * np.random.normal(size=len(y))
                feat_noise = extract_features_advanced(y_noise, sr)
                data_list.append(np.append(feat_noise, label))
                
                # Pitch Shift
                y_pitch = librosa.effects.pitch_shift(y=y, sr=sr, n_steps=2)
                feat_pitch = extract_features_advanced(y_pitch, sr)
                data_list.append(np.append(feat_pitch, label))

        except Exception as e:
            pass

# Save
cols = [f'f_{i}' for i in range(len(data_list[0])-1)] + ['label']
df = pd.DataFrame(data_list, columns=cols)
df.to_csv(os.path.join(OUTPUT_PATH, OUTPUT_FILE), index=False)

print(f"✅ Saved {len(df)} samples to {OUTPUT_FILE}")

f:\Research\Project\Final\infant-growth-monitoring-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Starting Advanced Feature Extraction...
   Processing hungry...
   Processing belly pain...
   Processing cold_hot...
   Processing discomfort...
   Processing tired...
   Processing burping...
   Processing lonely...
   Processing scared...
✅ Saved 2485 samples to Audio_CatBoost_Features.csv


In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier

# 1. Load Data
DATA_FILE = "../data/processed/Audio_CatBoost_Features.csv"
print("⏳ Loading Data...")
df = pd.read_csv(DATA_FILE)

# Clean infinite values just in case
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

X = df.drop('label', axis=1)
y = df['label']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training on {len(X_train)} samples...")

# 2. Define the 3 Models

# Model A: CatBoost (The "Paranoid" Expert)
clf1 = CatBoostClassifier(
    iterations=500, 
    learning_rate=0.05, 
    depth=6, 
    auto_class_weights='Balanced',
    verbose=0
)

# Model B: Random Forest (The "Stable" Expert)
clf2 = RandomForestClassifier(
    n_estimators=200, 
    class_weight='balanced', 
    random_state=42
)

# Model C: Logistic Regression (The "Strict" Expert)
clf3 = LogisticRegression(
    class_weight='balanced', 
    max_iter=1000, 
    solver='lbfgs'
)

# 3. Create the Voting Committee (Soft Voting = Average of probabilities)
print("🗳️  Training Voting Classifier (Combining 3 models)...")
voting_model = VotingClassifier(
    estimators=[
        ('cat', clf1), 
        ('rf', clf2), 
        ('lr', clf3)
    ],
    voting='soft' 
)

voting_model.fit(X_train, y_train)

# 4. Evaluate
print("\n🔍 Evaluation Results:")
y_pred = voting_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"🏆 Final Ensemble Accuracy: {acc*100:.2f}%")
print(classification_report(y_test, y_pred, target_names=['Hunger', 'Pain', 'Normal']))



⏳ Loading Data...
Training on 1988 samples...
🗳️  Training Voting Classifier (Combining 3 models)...

🔍 Evaluation Results:
🏆 Final Ensemble Accuracy: 42.86%
              precision    recall  f1-score   support

      Hunger       0.45      0.45      0.45       246
        Pain       0.37      0.42      0.40        80
      Normal       0.42      0.40      0.41       171

    accuracy                           0.43       497
   macro avg       0.42      0.43      0.42       497
weighted avg       0.43      0.43      0.43       497



f:\Research\Project\Final\infant-growth-monitoring-system\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
